Celda 1: Importación de librerías y carga de datos

In [1]:
import pandas as pd
import numpy as np
import os
import time
import warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from prophet import Prophet
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

warnings.filterwarnings('ignore')

PATH_MENSUAL = r"C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\dataset\dataset_mensual.xlsx"
PATH_SLIDING = r"C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\augmented\dataset_sliding_window.xlsx"
PATH_OUTPUT_DIR = r"C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\resultados"
PATH_GRAFICAS = r"C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\notebooks\prediction\graficas"

os.makedirs(PATH_OUTPUT_DIR, exist_ok=True)
os.makedirs(PATH_GRAFICAS, exist_ok=True)

df_mensual = pd.read_excel(PATH_MENSUAL)
if 'Periodo' in df_mensual.columns:
    df_mensual['Periodo'] = pd.to_datetime(df_mensual['Periodo'].astype(str))
    df_mensual = df_mensual.set_index('Periodo')

print(f"Dataset mensual: {len(df_mensual)} meses")
print(f"Columnas: {list(df_mensual.columns[:8])}")

Importing plotly failed. Interactive plots will not work.


Dataset mensual: 37 meses
Columnas: ['servicios_totales', 'monto_total', 'monto_promedio', 'monto_mediana', 'carroza_count', 'carroza_flores_count', 'auto_count', 'microbus_count']


Celda 2: Preparación de datos y split temporal

In [2]:
TARGETS = ['servicios_totales', 'monto_total']
test_size = 1

train = df_mensual.iloc[:-test_size]
test = df_mensual.iloc[-test_size:]

print(f"Train: {len(train)} meses ({train.index[0]} a {train.index[-1]})")
print(f"Test: {len(test)} meses ({test.index[0]} a {test.index[-1]})")
print(f"\nNOTA: Con solo {test_size} punto(s) de test, las metricas son indicativas, no definitivas.")

Train: 36 meses (2022-08-01 00:00:00 a 2025-07-01 00:00:00)
Test: 1 meses (2025-08-01 00:00:00 a 2025-08-01 00:00:00)

NOTA: Con solo 1 punto(s) de test, las metricas son indicativas, no definitivas.


Celda 3: SARIMA

In [3]:
def entrenar_evaluar_sarima(y_train, y_test):
    start = time.time()
    model = SARIMAX(y_train, order=(1,1,1), seasonal_order=(0,0,0,0), enforce_stationarity=False, enforce_invertibility=False)
    results = model.fit(disp=False)
    pred = results.forecast(steps=len(y_test))
    tiempo = time.time() - start
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    mape = np.mean(np.abs((y_test.values - pred.values) / np.where(y_test.values == 0, 1, y_test.values))) * 100
    return {
        'modelo': 'SARIMA',
        'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape,
        'tiempo_entrenamiento': tiempo,
        'predicciones': pred.tolist(), 'real': y_test.tolist()
    }

resultados = []
for target in TARGETS:
    y_train = train[target]
    y_test = test[target]
    res = entrenar_evaluar_sarima(y_train, y_test)
    res['target'] = target
    resultados.append(res)
    print(f"SARIMA - {target}: MAE={res['MAE']:.2f}, R2={res['R2']:.4f}, Tiempo={res['tiempo_entrenamiento']:.2f}s")

SARIMA - servicios_totales: MAE=4.24, R2=nan, Tiempo=0.03s
SARIMA - monto_total: MAE=8562.52, R2=nan, Tiempo=0.02s


Celda 4: Prophet

In [4]:
def entrenar_evaluar_prophet(y_train, y_test):
    start = time.time()
    df_p = pd.DataFrame({'ds': y_train.index, 'y': y_train.values})
    model = Prophet(yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False, changepoint_prior_scale=0.5)
    model.fit(df_p)
    future = pd.DataFrame({'ds': y_test.index})
    forecast = model.predict(future)
    pred = forecast['yhat'].values[:len(y_test)]
    tiempo = time.time() - start
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    mape = np.mean(np.abs((y_test.values - pred) / np.where(y_test.values == 0, 1, y_test.values))) * 100
    return {
        'modelo': 'Prophet',
        'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape,
        'tiempo_entrenamiento': tiempo,
        'predicciones': pred.tolist(), 'real': y_test.tolist()
    }

for target in TARGETS:
    y_train = train[target]
    y_test = test[target]
    res = entrenar_evaluar_prophet(y_train, y_test)
    res['target'] = target
    resultados.append(res)
    print(f"Prophet - {target}: MAE={res['MAE']:.2f}, R2={res['R2']:.4f}, Tiempo={res['tiempo_entrenamiento']:.2f}s")

09:54:45 - cmdstanpy - INFO - Chain [1] start processing


09:54:47 - cmdstanpy - INFO - Chain [1] done processing


09:54:47 - cmdstanpy - INFO - Chain [1] start processing


Prophet - servicios_totales: MAE=4.20, R2=nan, Tiempo=2.45s


09:54:47 - cmdstanpy - INFO - Chain [1] done processing


Prophet - monto_total: MAE=10573.39, R2=nan, Tiempo=0.38s


Celda 5: XGBoost con Lags

In [5]:
def entrenar_evaluar_xgboost(df_windows, target):
    start = time.time()
    lag_cols = [f'{target}_lag_1', f'{target}_lag_2', f'{target}_lag_3']
    target_col = f'{target}_target'

    if not all(c in df_windows.columns for c in lag_cols + [target_col]):
        return None

    X = df_windows[lag_cols]
    y = df_windows[target_col]

    if len(X) < 2:
        return None

    split_idx = max(1, int(len(X) * 0.8))
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    model = XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42, verbosity=0)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    tiempo = time.time() - start

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred) if len(y_test) > 1 else 0.0
    mape = np.mean(np.abs((y_test.values - pred) / np.where(y_test.values == 0, 1, y_test.values))) * 100

    return {
        'modelo': 'XGBoost', 'target': target,
        'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape,
        'tiempo_entrenamiento': tiempo,
        'predicciones': pred.tolist(), 'real': y_test.tolist()
    }

df_windows = pd.read_excel(PATH_SLIDING)

for target in TARGETS:
    res = entrenar_evaluar_xgboost(df_windows, target)
    if res:
        resultados.append(res)
        print(f"XGBoost - {target}: MAE={res['MAE']:.2f}, R2={res['R2']:.4f}, Tiempo={res['tiempo_entrenamiento']:.2f}s")

XGBoost - servicios_totales: MAE=1.74, R2=-0.3898, Tiempo=1.13s
XGBoost - monto_total: MAE=6023.42, R2=-0.3825, Tiempo=0.04s


Celda 6: LightGBM con Lags

In [6]:
def entrenar_evaluar_lgbm(df_windows, target):
    start = time.time()
    lag_cols = [f'{target}_lag_1', f'{target}_lag_2', f'{target}_lag_3']
    target_col = f'{target}_target'

    if not all(c in df_windows.columns for c in lag_cols + [target_col]):
        return None

    X = df_windows[lag_cols]
    y = df_windows[target_col]

    if len(X) < 2:
        return None

    split_idx = max(1, int(len(X) * 0.8))
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    model = LGBMRegressor(num_leaves=15, n_estimators=100, learning_rate=0.05, random_state=42, verbose=-1)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    tiempo = time.time() - start

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred) if len(y_test) > 1 else 0.0
    mape = np.mean(np.abs((y_test.values - pred) / np.where(y_test.values == 0, 1, y_test.values))) * 100

    return {
        'modelo': 'LightGBM', 'target': target,
        'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape,
        'tiempo_entrenamiento': tiempo,
        'predicciones': pred.tolist(), 'real': y_test.tolist()
    }

for target in TARGETS:
    res = entrenar_evaluar_lgbm(df_windows, target)
    if res:
        resultados.append(res)
        print(f"LightGBM - {target}: MAE={res['MAE']:.2f}, R2={res['R2']:.4f}, Tiempo={res['tiempo_entrenamiento']:.2f}s")

LightGBM - servicios_totales: MAE=1.73, R2=-0.3829, Tiempo=1.65s
LightGBM - monto_total: MAE=6007.78, R2=-0.3798, Tiempo=0.00s


Celda 7: LSTM

In [7]:
def entrenar_evaluar_lstm(y_train, y_test, window_size=3):
    start = time.time()

    scaler = MinMaxScaler()
    y_scaled = scaler.fit_transform(y_train.values.reshape(-1, 1))

    if len(y_scaled) < window_size:
        return None

    X_train_lstm = []
    y_train_lstm = []
    for i in range(window_size, len(y_scaled)):
        X_train_lstm.append(y_scaled[i-window_size:i, 0])
        y_train_lstm.append(y_scaled[i, 0])
    X_train_lstm = np.array(X_train_lstm)
    y_train_lstm = np.array(y_train_lstm)
    X_train_lstm = X_train_lstm.reshape(X_train_lstm.shape[0], X_train_lstm.shape[1], 1)

    model = Sequential([
        LSTM(32, input_shape=(window_size, 1)),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    model.fit(X_train_lstm, y_train_lstm, epochs=50, batch_size=4, verbose=0)

    last_sequence = y_scaled[-window_size:].reshape(1, window_size, 1)
    pred_scaled = model.predict(last_sequence, verbose=0)
    pred = scaler.inverse_transform(pred_scaled)[0, 0]
    tiempo = time.time() - start

    y_true = y_test.values[0] if len(y_test) > 0 else 0
    mae = abs(y_true - pred)
    rmse = mae
    r2 = 0.0
    mape = abs((y_true - pred) / y_true) * 100 if y_true != 0 else 0

    return {
        'modelo': 'LSTM', 'target': '',
        'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape,
        'tiempo_entrenamiento': tiempo,
        'predicciones': [pred], 'real': [y_true]
    }

for target in TARGETS:
    y_train = train[target]
    y_test = test[target]
    res = entrenar_evaluar_lstm(y_train, y_test)
    if res:
        res['target'] = target
        resultados.append(res)
        print(f"LSTM - {target}: MAE={res['MAE']:.2f}, R2={res['R2']:.4f}, Tiempo={res['tiempo_entrenamiento']:.2f}s")

LSTM - servicios_totales: MAE=7.06, R2=0.0000, Tiempo=5.66s


LSTM - monto_total: MAE=24925.62, R2=0.0000, Tiempo=4.83s


Celda 8: ETS (Exponential Smoothing)

In [8]:
def entrenar_evaluar_ets(y_train, y_test):
    start = time.time()
    try:
        model = ExponentialSmoothing(y_train, trend='add', seasonal=None, initialization_method='estimated')
        results = model.fit(optimized=True)
        pred = results.forecast(steps=len(y_test))
    except Exception:
        model = ExponentialSmoothing(y_train, trend='add', seasonal=None, initialization_method='heuristic')
        results = model.fit(optimized=True)
        pred = results.forecast(steps=len(y_test))
    tiempo = time.time() - start
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    mape = np.mean(np.abs((y_test.values - pred.values) / np.where(y_test.values == 0, 1, y_test.values))) * 100
    return {
        'modelo': 'ETS',
        'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape,
        'tiempo_entrenamiento': tiempo,
        'predicciones': pred.tolist(), 'real': y_test.tolist()
    }

for target in TARGETS:
    y_train = train[target]
    y_test = test[target]
    res = entrenar_evaluar_ets(y_train, y_test)
    res['target'] = target
    resultados.append(res)
    print(f"ETS - {target}: MAE={res['MAE']:.2f}, R2={res['R2']:.4f}, Tiempo={res['tiempo_entrenamiento']:.2f}s")

ETS - servicios_totales: MAE=4.91, R2=nan, Tiempo=0.01s
ETS - monto_total: MAE=13394.93, R2=nan, Tiempo=0.02s


Celda 9: Tabla Comparativa y Visualizaciones

In [9]:
df_comparativa = pd.DataFrame(resultados)
df_comparativa = df_comparativa[['modelo', 'target', 'MAE', 'RMSE', 'R2', 'MAPE', 'tiempo_entrenamiento']]

print("Tabla Comparativa:")
print(df_comparativa.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, metric in enumerate(['MAE', 'RMSE', 'R2']):
    ax = axes[i]
    for target in TARGETS:
        subset = df_comparativa[df_comparativa['target'] == target]
        ax.bar(subset['modelo'] + ' (' + target[:4] + ')', subset[metric], label=target)
    ax.set_title(metric)
    ax.tick_params(axis='x', rotation=45)
    ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PATH_GRAFICAS, 'comparativa_mae_rmse.png'), dpi=150, bbox_inches='tight')
plt.close()

print(f"\nGrafico comparativo guardado en: {PATH_GRAFICAS}")

Tabla Comparativa:
  modelo            target          MAE         RMSE        R2        MAPE  tiempo_entrenamiento
  SARIMA servicios_totales     4.243680     4.243680       NaN   70.728002              0.029503
  SARIMA       monto_total  8562.515218  8562.515218       NaN   47.996162              0.018857
 Prophet servicios_totales     4.199077     4.199077       NaN   69.984617              2.451059
 Prophet       monto_total 10573.391078 10573.391078       NaN   59.267887              0.380970
 XGBoost servicios_totales     1.738841     3.195433 -0.389802   31.558445              1.129066
 XGBoost       monto_total  6023.422852 11285.844231 -0.382521 5463.274383              0.039748
LightGBM servicios_totales     1.730159     3.187491 -0.382903   31.040564              1.647753
LightGBM       monto_total  6007.777778 11274.681563 -0.379787 4658.110608              0.000000
    LSTM servicios_totales     7.060318     7.060318  0.000000  117.671967              5.659847
    LSTM   


Grafico comparativo guardado en: C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\notebooks\prediction\graficas


Celda 10: Visualizaciones por Modelo

In [10]:
modelos_nombres = {
    'SARIMA': 'sarima', 'Prophet': 'prophet', 'XGBoost': 'xgboost',
    'LightGBM': 'lgbm', 'LSTM': 'lstm', 'ETS': 'ets'
}

for target in TARGETS:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()

    target_results = [r for r in resultados if r['target'] == target]

    for idx, res in enumerate(target_results):
        if idx >= len(axes):
            break
        ax = axes[idx]
        modelo_key = modelos_nombres.get(res['modelo'], res['modelo'].lower())

        real_vals = res['real']
        pred_vals = res['predicciones']
        x_real = range(len(real_vals))
        x_pred = range(len(pred_vals))

        ax.plot(x_real, real_vals, 'b-o', label='Real', markersize=5)
        ax.plot(x_pred, pred_vals, 'r--x', label='Predicho', markersize=5)
        ax.set_title(f"{res['modelo']} - {target}\nMAE={res['MAE']:.2f}, R2={res['R2']:.4f}")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    for idx in range(len(target_results), len(axes)):
        axes[idx].set_visible(False)

    plt.single_axes_title = f"Predicciones por Modelo - {target}"
    plt.tight_layout()
    plt.savefig(os.path.join(PATH_GRAFICAS, f'predicciones_{target}.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Grafico guardado: predicciones_{target}.png")

Grafico guardado: predicciones_servicios_totales.png


Grafico guardado: predicciones_monto_total.png


Celda 11: Exportación de Resultados

In [11]:
df_comparativa.to_excel(os.path.join(PATH_OUTPUT_DIR, 'comparativa_modelos.xlsx'), index=False)

predicciones_rows = []
for res in resultados:
    for i, (real, pred) in enumerate(zip(res['real'], res['predicciones'])):
        predicciones_rows.append({
            'modelo': res['modelo'],
            'target': res['target'],
            'punto': i,
            'real': real,
            'predicho': pred,
            'error_absoluto': abs(real - pred)
        })

df_predicciones = pd.DataFrame(predicciones_rows)
df_predicciones.to_excel(os.path.join(PATH_OUTPUT_DIR, 'predicciones_todos_modelos.xlsx'), index=False)

print(f"Archivos exportados en: {PATH_OUTPUT_DIR}")
print("Pipeline de modelos temporales completado.")

Archivos exportados en: C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\resultados
Pipeline de modelos temporales completado.
